In [3]:
!pip install nltk --quiet

In [11]:
import re 
import math 
from collections import Counter 
from typing import List, Tuple, Dict 
import matplotlib.pyplot as plt 
import random 
random.seed(13)

In [12]:
def simple_tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

In [13]:
def add_sentence_markers(tokens: List[str], n:int) -> List[str]:
    k = n - 1
    return (['<s>'] * k + tokens + ['</s>']) 
def ngrams(seq: List[str], n:int) -> List[Tuple[str, ...]]:
    return [ tuple(seq[i:i+n]) for i in range(len(seq) - n + 1)]
def train_test_split(tokens: List[str], test_ratio = 0.2):
    split_index = int(len(tokens) * (1 - test_ratio))
    return tokens[:split_index], tokens[split_index:]
print("setup complete")


setup complete


In [14]:
toy_text = """
I love NLP and I love building language models.
Language models help phones suggest the next word.
I am going to the store to buy some apples.
I am going to the gym after work.
We love natural language processing and we study NLP.
Unicorns are rare but language is creative and flexible.
"""
#tokenize to lower words
tokens = simple_tokenize(toy_text)
print(tokens[:30])
print("Total tokens:", len(tokens))

['i', 'love', 'nlp', 'and', 'i', 'love', 'building', 'language', 'models', 'language', 'models', 'help', 'phones', 'suggest', 'the', 'next', 'word', 'i', 'am', 'going', 'to', 'the', 'store', 'to', 'buy', 'some', 'apples', 'i', 'am', 'going']
Total tokens: 53


In [17]:
def build_vocab(tokens: List[str], min_frq = 1) -> Dict[str, int]:
    freqs = Counter(tokens)
    vocab = {"<unk>": 0, "<s>":1, "</s>":2}
    for w,c in freqs.items():
        if c >= min_frq and w not in vocab:
            vocab[w] = len(vocab) 
    return vocab 
def map_to_vocab(tokens: List[str], vocab: Dict[str, int]) -> List[str]:
    return [w if w in vocab else "<unk>" for w in tokens]

def build_ngram_counts(tokens: List[str], n:int) -> Counter:
    return Counter(ngrams(tokens, n))
# Build vocab and split data
vocab = build_vocab(tokens, min_frq = 1) 
train_toks = tokens 
test_toks = []
#Prepare sequences with sentence markers 
N = 3 
train_seq = add_sentence_markers(map_to_vocab(train_toks, vocab), N) 
test_seq = add_sentence_markers(map_to_vocab(test_toks, vocab), N)
#count unigram/ bigram/ trigram
uni_counts = build_ngram_counts(train_seq, 1)
bi_counts = build_ngram_counts(train_seq, 2)
tri_counts = build_ngram_counts(train_seq, 3)

V = len(vocab)
print("Vocabulary size:", V)
print("train_toksoks", train_seq)
print("Top unigrams:", uni_counts.most_common(8)) 
print("Top bigrams:", bi_counts.most_common(8)) 
print("Top trigrams:", tri_counts.most_common(8)) 

Vocabulary size: 37
train_toksoks ['<s>', '<s>', 'i', 'love', 'nlp', 'and', 'i', 'love', 'building', 'language', 'models', 'language', 'models', 'help', 'phones', 'suggest', 'the', 'next', 'word', 'i', 'am', 'going', 'to', 'the', 'store', 'to', 'buy', 'some', 'apples', 'i', 'am', 'going', 'to', 'the', 'gym', 'after', 'work', 'we', 'love', 'natural', 'language', 'processing', 'and', 'we', 'study', 'nlp', 'unicorns', 'are', 'rare', 'but', 'language', 'is', 'creative', 'and', 'flexible', '</s>']
Top unigrams: [(('i',), 4), (('language',), 4), (('love',), 3), (('and',), 3), (('the',), 3), (('to',), 3), (('<s>',), 2), (('nlp',), 2)]
Top bigrams: [(('i', 'love'), 2), (('language', 'models'), 2), (('i', 'am'), 2), (('am', 'going'), 2), (('going', 'to'), 2), (('to', 'the'), 2), (('<s>', '<s>'), 1), (('<s>', 'i'), 1)]
Top trigrams: [(('i', 'am', 'going'), 2), (('am', 'going', 'to'), 2), (('going', 'to', 'the'), 2), (('<s>', '<s>', 'i'), 1), (('<s>', 'i', 'love'), 1), (('i', 'love', 'nlp'), 1), 

In [19]:
def bigram_mle_prob(w_prev: str, w:str) -> float:
    "P(w  | w_prev) = count(w_prev, w) / count(w_prev) "
    "it returns 0 if denominator is 0 (unseen history)" 
    c_bigram = bi_counts.get((w_prev, w), 0)
    c_prev = uni_counts.get((w_prev,), 0)
    return (c_bigram/ c_prev) if c_prev > 0 else 0.0 
def trigram_mle_prob(w1: str, w2: str, w3: str):
    c_trigram = tri_counts.get((w1, w2, w3), 0)
    c_hist = bi_counts.get((w1, w2), 0)
    return (c_trigram/ c_hist) if c_hist > 0 else 0.0 
def sentence_logprob_mle(sent_tokens: List[str], n = 3) -> float:
    seq = add_sentence_markers(map_to_vocab(sent_tokens, vocab, ), n) 
    logp = 0.0 
    for (w1, w2, w3) in ngrams(seq, 3):
        p = trigram_mle_prob(w1, w2, w3)
        if p == 0.0:
            return float("-inf")
        logp += math.log(p)
    return logp
example = simple_tokenize("I saw a unicorn today") 
print("Trigram MLE long-prob:", sentence_logprob_mle(example, n = 3))


Trigram MLE long-prob: -inf


# Add 1 or k smoothing

In [25]:
def bigram_addk_prob(w_prev: str, w: str, k : float) -> float:
    number = bi_counts.get((w_prev, w), 0) + k 
    denom = uni_counts.get((w_prev,), 0) + k * v 
    return (number/ denom) if denom > 0 else (1.0/ V)
def trigram_addk_prob(w1:str, w2:str, w3:str, k: float):
    number = tri_counts.get((w1, w2, w3), 0) + k 
    denom = bi_counts.get((w1,w2), 0) + k * V 
    return (number / denom) if denom > 0 else (1.0/ V)
def sentence_logprob_addk(sent_tokens: List[str], n = 3, k = 1.0) -> float:
    seq = add_sentence_markers(map_to_vocab(sent_tokens, vocab), n)
    logp = 0.0 
    for (w1, w2, w3) in ngrams(seq,3):
        p = trigram_addk_prob(w1, w2, w3, k)
        logp += math.log(max(p, 1e-12))
    return logp 
for k in [0.0, 0.1, 0.5, 1.0]:
    print(f"Add-k (k = {k} trigram log-prob)", sentence_logprob_addk(example,n = 3,  k=k))





Add-k (k = 0.0 trigram log-prob) -42.07469276650545
Add-k (k = 0.1 trigram log-prob) -19.746071581198645
Add-k (k = 0.5 trigram log-prob) -20.672182654168083
Add-k (k = 1.0 trigram log-prob) -21.025696789469723
